In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================
# SPLIT EXTRACTED FRAMES INTO TRAIN / VAL
# Shuffles frames within each class before split
# Test set will be collected separately later
# ============================================

import os
import random
import shutil
from pathlib import Path

In [3]:
# -----------------------------
# PATHS
# -----------------------------
BASE_DIR = "/content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization"

EXTRACTED_FRAMES_DIR = os.path.join(BASE_DIR, "extracted_frames_test")
PROCESSED_IMAGES_DIR = os.path.join(BASE_DIR, "processed_images")


TEST_DIR = os.path.join(PROCESSED_IMAGES_DIR, "test")


os.makedirs(TEST_DIR, exist_ok=True)

print("EXTRACTED_FRAMES_DIR:", EXTRACTED_FRAMES_DIR)
print("TEST_DIR:", TEST_DIR)


EXTRACTED_FRAMES_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/extracted_frames_test
TEST_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/processed_images/test


In [4]:
# -----------------------------
# CLASS NAMES
# -----------------------------
class_names = [
    "floor1_hallway",
    "floor2_hallway",
    "floor3_hallway",
    "front_lobby",
    "laundry",
    "ccu_lounge",
    "floor1_elevator_landmark",
    "floor2_elevator_landmark",
    "floor3_elevator_landmark"
]

In [5]:
# -----------------------------
# SETTINGS
# -----------------------------
test_ratio = 0.7 #=-> ONLY DOING THIS SO I CAN REUSE FILE AND ALSO BECAUSE I WILL USE TEST RECORDINGS FOR DEMO AS WELL SO WANT TO REATIN SOME ORIGINAL CLIPS
random_seed = 26

image_extensions = {".jpg", ".jpeg", ".png"}
random.seed(random_seed)

In [6]:
def get_image_files(folder_path):
    return [
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if Path(f).suffix.lower() in image_extensions
    ]

In [8]:
# -----------------------------
# SPLIT FRAMES FOR EACH CLASS
# -----------------------------
summary = {}

for class_name in class_names:
    class_input_dir = os.path.join(EXTRACTED_FRAMES_DIR, class_name)
    class_test_dir = os.path.join(TEST_DIR, class_name)

    os.makedirs(class_test_dir, exist_ok=True)

    if not os.path.exists(class_input_dir):
        print(f"[WARNING] Missing extracted frames folder: {class_input_dir}")
        summary[class_name] = {"total": 0, "test": 0}
        continue

    image_files = get_image_files(class_input_dir)

    # Shuffle before splitting
    random.shuffle(image_files)

    total_count = len(image_files)
    test_count = int(total_count * test_ratio)

    # Corrected: Take the first 'test_count' files for the test set
    test_files = image_files[:test_count]

    # Corrected: Copy test_files into the test folder
    for src_path in test_files:
        dst_path = os.path.join(class_test_dir, os.path.basename(src_path))
        shutil.copy2(src_path, dst_path)


    summary[class_name] = {
        "total": total_count,
        "test": len(test_files),
    }

print("\n=== TEST SPLIT SUMMARY ===")
for class_name, counts in summary.items():
    print(
        f"{class_name}: total={counts['total']}, "
        f"test={counts['test']}"
    )


=== TEST SPLIT SUMMARY ===
floor1_hallway: total=51, test=35
floor2_hallway: total=304, test=212
floor3_hallway: total=144, test=100
front_lobby: total=70, test=49
laundry: total=65, test=45
ccu_lounge: total=70, test=49
floor1_elevator_landmark: total=47, test=32
floor2_elevator_landmark: total=72, test=50
floor3_elevator_landmark: total=52, test=36


In [9]:
# -----------------------------
# FINAL COUNT CHECK
# -----------------------------
print("\n=== FINAL FOLDER COUNTS ===")

for split_name, split_dir in [("test", TEST_DIR)]:
    print(f"\n{split_name.upper()}:")
    for class_name in class_names:
        class_dir = os.path.join(split_dir, class_name)
        if os.path.exists(class_dir):
            count = len(get_image_files(class_dir))
            print(f"  {class_name}: {count}")
        else:
            print(f"  {class_name}: 0")


=== FINAL FOLDER COUNTS ===

TEST:
  floor1_hallway: 35
  floor2_hallway: 212
  floor3_hallway: 100
  front_lobby: 49
  laundry: 45
  ccu_lounge: 49
  floor1_elevator_landmark: 32
  floor2_elevator_landmark: 50
  floor3_elevator_landmark: 36
